# SRGL Ablation Study

This notebook performs a comprehensive ablation study to evaluate the contribution of each gate and design component.

## Overview
- Test SRGL with different gate combinations
- Evaluate impact of each gate on performance
- Compare merging strategies
- Analyze gate activation patterns
- Statistical significance testing

## Requirements
Run notebook 01 first to generate the datasets.

In [ ]:
import sys
import os

# Add parent directory to path
sys.path.append(os.path.abspath('..'))

import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from typing import List, Dict, Set
from itertools import combinations

from src.data_generator import PatientCase, RiskTier
from src.gates import (
    Gate_G1_CriticalFlags,
    Gate_G2_ModerateRisk,
    Gate_G3_DataQuality,
    Gate_G4_TiTrATE,
    Gate_G5_Uncertainty,
    Gate_G6_Temporal
)
from src.merging import ConservativeMerging, AverageMerging, WeightedMerging
from src.evaluation import SafetyMetrics, StatisticalTests

# Set random seed
np.random.seed(42)

# Configure plotting
plt.style.use('seaborn-v0_8-paper')
sns.set_palette('colorblind')
%matplotlib inline

## 1. Load Test Dataset

In [ ]:
def load_cases(filepath: str) -> List[PatientCase]:
    """Load patient cases from JSON file."""
    with open(filepath, 'r') as f:
        data = json.load(f)
    
    cases = []
    for case_data in data['cases']:
        case = PatientCase(
            case_id=case_data['case_id'],
            diagnosis=case_data['diagnosis'],
            ground_truth_tier=RiskTier(case_data['ground_truth_tier']),
            patient_features=case_data['patient_features'],
            clinical_presentation=type('obj', (object,), case_data['clinical_presentation'])(),
            vital_signs=case_data.get('vital_signs', {}),
            lab_results=case_data.get('lab_results', {}),
            timestamp=case_data.get('timestamp', '')
        )
        cases.append(case)
    
    return cases

# Load test dataset
test_cases = load_cases('../data/synthetic_test.json')
print(f"Loaded {len(test_cases)} test cases")

## 2. Define Ablation Configurations

Test different combinations of gates to understand their individual contributions.

In [ ]:
# Define all gates
ALL_GATES = {
    'G1': Gate_G1_CriticalFlags(),
    'G2': Gate_G2_ModerateRisk(),
    'G3': Gate_G3_DataQuality(),
    'G4': Gate_G4_TiTrATE(),
    'G5': Gate_G5_Uncertainty(),
    'G6': Gate_G6_Temporal()
}

# Define ablation configurations
ABLATION_CONFIGS = {
    'Full SRGL': ['G1', 'G2', 'G3', 'G4', 'G5', 'G6'],
    'Without G1 (Critical)': ['G2', 'G3', 'G4', 'G5', 'G6'],
    'Without G2 (Moderate)': ['G1', 'G3', 'G4', 'G5', 'G6'],
    'Without G3 (Data Quality)': ['G1', 'G2', 'G4', 'G5', 'G6'],
    'Without G4 (TiTrATE)': ['G1', 'G2', 'G3', 'G5', 'G6'],
    'Without G5 (Uncertainty)': ['G1', 'G2', 'G3', 'G4', 'G6'],
    'Without G6 (Temporal)': ['G1', 'G2', 'G3', 'G4', 'G5'],
    'Traditional Only (G1,G2,G4)': ['G1', 'G2', 'G4'],
    'Novel Only (G3,G5,G6)': ['G3', 'G5', 'G6'],
    'Minimal (G1,G2)': ['G1', 'G2']
}

print("Ablation Configurations:")
for name, gates in ABLATION_CONFIGS.items():
    print(f"  {name}: {', '.join(gates)}")

## 3. Run Ablation Experiments

In [ ]:
def run_ablation_experiment(cases: List[PatientCase], 
                           active_gates: List[str],
                           merging_strategy='conservative') -> pd.DataFrame:
    """Run experiment with specified gate configuration."""
    
    # Initialize merging
    if merging_strategy == 'conservative':
        merger = ConservativeMerging()
    elif merging_strategy == 'average':
        merger = AverageMerging()
    else:
        merger = WeightedMerging()
    
    results = []
    
    for case in cases:
        # Convert case to input format
        patient_data = {
            'patient_id': case.case_id,
            'age': case.patient_features['age'],
            'sex': case.patient_features['sex'],
            'symptoms': case.clinical_presentation.symptoms,
            'red_flags': case.clinical_presentation.red_flags,
            'risk_factors': case.clinical_presentation.risk_factors,
            'vital_signs': case.vital_signs,
            'lab_results': case.lab_results,
            'timestamp': case.timestamp
        }
        
        # Run active gates
        gate_outputs = []
        for gate_name in active_gates:
            gate = ALL_GATES[gate_name]
            output = gate.evaluate(patient_data)
            gate_outputs.append(output)
        
        # Merge outputs
        decision = merger.merge(gate_outputs)
        
        results.append({
            'case_id': case.case_id,
            'ground_truth': case.ground_truth_tier.value,
            'predicted': decision.final_tier.value if decision.final_tier else 0,
            'is_abstention': decision.final_tier is None
        })
    
    return pd.DataFrame(results)

# Run all ablation experiments
print("Running ablation experiments...\n")
ablation_results = {}

for config_name, gates in ABLATION_CONFIGS.items():
    print(f"Testing: {config_name}")
    results = run_ablation_experiment(test_cases, gates, merging_strategy='conservative')
    ablation_results[config_name] = results
    print(f"  Completed: {len(results)} predictions\n")

print("All experiments completed!")

## 4. Calculate Metrics for Each Configuration

In [ ]:
# Calculate metrics for all configurations
metrics_summary = []

for config_name, results in ablation_results.items():
    y_true = results['ground_truth'].values
    y_pred = results['predicted'].values
    
    metrics = SafetyMetrics(y_true, y_pred)
    
    sensitivity = metrics.sensitivity_critical(critical_threshold=3)
    specificity = metrics.specificity_safe(safe_threshold=1)
    fnr = metrics.false_negative_rate(critical_threshold=3)
    unsafe_discharge = metrics.unsafe_discharge_rate(discharge_threshold=1)
    abstention = metrics.abstention_rate()
    
    metrics_summary.append({
        'Configuration': config_name,
        'Sensitivity': sensitivity['sensitivity'],
        'Sensitivity_CI': f"[{sensitivity['ci_lower']:.3f}, {sensitivity['ci_upper']:.3f}]",
        'Specificity': specificity['specificity'],
        'Specificity_CI': f"[{specificity['ci_lower']:.3f}, {specificity['ci_upper']:.3f}]",
        'FNR': fnr['fnr'],
        'FNR_CI': f"[{fnr['ci_lower']:.3f}, {fnr['ci_upper']:.3f}]",
        'Unsafe_Discharge': unsafe_discharge['unsafe_discharge_rate'],
        'Unsafe_CI': f"[{unsafe_discharge['ci_lower']:.3f}, {unsafe_discharge['ci_upper']:.3f}]",
        'Abstention': abstention['abstention_rate'],
        'Abstention_CI': f"[{abstention['ci_lower']:.3f}, {abstention['ci_upper']:.3f}]"
    })

# Create summary DataFrame
ablation_summary = pd.DataFrame(metrics_summary)

print("\n" + "="*80)
print("ABLATION STUDY RESULTS")
print("="*80)
print(ablation_summary[['Configuration', 'Sensitivity', 'Specificity', 'FNR', 'Abstention']].to_string(index=False))
print("="*80)

## 5. Visualize Ablation Results

In [ ]:
# Create comprehensive visualization
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Sort by sensitivity
ablation_sorted = ablation_summary.sort_values('Sensitivity', ascending=True)

# 1. Sensitivity comparison
y_pos = np.arange(len(ablation_sorted))
axes[0, 0].barh(y_pos, ablation_sorted['Sensitivity'].values, color='steelblue')
axes[0, 0].set_yticks(y_pos)
axes[0, 0].set_yticklabels(ablation_sorted['Configuration'].values, fontsize=9)
axes[0, 0].set_xlabel('Sensitivity', fontsize=12)
axes[0, 0].set_title('Sensitivity by Configuration', fontsize=14, fontweight='bold')
axes[0, 0].axvline(ablation_summary[ablation_summary['Configuration'] == 'Full SRGL']['Sensitivity'].values[0],
                   color='red', linestyle='--', linewidth=2, label='Full SRGL')
axes[0, 0].legend()
axes[0, 0].grid(axis='x', alpha=0.3)

# 2. Specificity comparison
ablation_sorted_spec = ablation_summary.sort_values('Specificity', ascending=True)
y_pos = np.arange(len(ablation_sorted_spec))
axes[0, 1].barh(y_pos, ablation_sorted_spec['Specificity'].values, color='seagreen')
axes[0, 1].set_yticks(y_pos)
axes[0, 1].set_yticklabels(ablation_sorted_spec['Configuration'].values, fontsize=9)
axes[0, 1].set_xlabel('Specificity', fontsize=12)
axes[0, 1].set_title('Specificity by Configuration', fontsize=14, fontweight='bold')
axes[0, 1].axvline(ablation_summary[ablation_summary['Configuration'] == 'Full SRGL']['Specificity'].values[0],
                   color='red', linestyle='--', linewidth=2, label='Full SRGL')
axes[0, 1].legend()
axes[0, 1].grid(axis='x', alpha=0.3)

# 3. FNR comparison
ablation_sorted_fnr = ablation_summary.sort_values('FNR', ascending=False)
y_pos = np.arange(len(ablation_sorted_fnr))
axes[1, 0].barh(y_pos, ablation_sorted_fnr['FNR'].values, color='coral')
axes[1, 0].set_yticks(y_pos)
axes[1, 0].set_yticklabels(ablation_sorted_fnr['Configuration'].values, fontsize=9)
axes[1, 0].set_xlabel('False Negative Rate', fontsize=12)
axes[1, 0].set_title('False Negative Rate by Configuration', fontsize=14, fontweight='bold')
axes[1, 0].axvline(ablation_summary[ablation_summary['Configuration'] == 'Full SRGL']['FNR'].values[0],
                   color='red', linestyle='--', linewidth=2, label='Full SRGL')
axes[1, 0].legend()
axes[1, 0].grid(axis='x', alpha=0.3)

# 4. Abstention rate comparison
ablation_sorted_abs = ablation_summary.sort_values('Abstention', ascending=True)
y_pos = np.arange(len(ablation_sorted_abs))
axes[1, 1].barh(y_pos, ablation_sorted_abs['Abstention'].values, color='mediumpurple')
axes[1, 1].set_yticks(y_pos)
axes[1, 1].set_yticklabels(ablation_sorted_abs['Configuration'].values, fontsize=9)
axes[1, 1].set_xlabel('Abstention Rate', fontsize=12)
axes[1, 1].set_title('Abstention Rate by Configuration', fontsize=14, fontweight='bold')
axes[1, 1].axvline(ablation_summary[ablation_summary['Configuration'] == 'Full SRGL']['Abstention'].values[0],
                   color='red', linestyle='--', linewidth=2, label='Full SRGL')
axes[1, 1].legend()
axes[1, 1].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('../figures/ablation_study_comprehensive.png', dpi=300, bbox_inches='tight')
plt.show()

print("Figure saved: figures/ablation_study_comprehensive.png")

## 6. Statistical Significance Testing

Test if removing each gate significantly affects performance.

In [ ]:
# Compare each configuration against Full SRGL
full_srgl_results = ablation_results['Full SRGL']
y_true = full_srgl_results['ground_truth'].values
y_pred_full = full_srgl_results['predicted'].values

print("\n" + "="*80)
print("STATISTICAL SIGNIFICANCE TESTING (McNemar's Test)")
print("="*80)

significance_tests = []

for config_name, results in ablation_results.items():
    if config_name == 'Full SRGL':
        continue
    
    y_pred_ablated = results['predicted'].values
    
    # McNemar's test
    test_result = StatisticalTests.mcnemar_test(
        y_true=y_true,
        y_pred1=y_pred_full,
        y_pred2=y_pred_ablated,
        critical_threshold=3
    )
    
    significance_tests.append({
        'Configuration': config_name,
        'Statistic': test_result['statistic'],
        'P-value': test_result['p_value'],
        'Significant': 'Yes' if test_result['significant'] else 'No',
        'Interpretation': test_result['interpretation']
    })
    
    print(f"\n{config_name}:")
    print(f"  Statistic: {test_result['statistic']:.3f}")
    print(f"  P-value: {test_result['p_value']:.4f}")
    print(f"  Significant: {test_result['significant']}")
    print(f"  {test_result['interpretation']}")

# Apply Bonferroni correction
p_values = [t['P-value'] for t in significance_tests]
corrected = StatisticalTests.bonferroni_correction(p_values, alpha=0.05)

print("\n" + "="*80)
print("BONFERRONI CORRECTION")
print("="*80)
print(f"Original alpha: 0.05")
print(f"Adjusted alpha: {corrected['adjusted_alpha']:.4f}")
print(f"Significant comparisons: {corrected['significant']}")
print("="*80)

# Save results
significance_df = pd.DataFrame(significance_tests)
significance_df.to_csv('../results/ablation_significance_tests.csv', index=False)
print("\nResults saved: results/ablation_significance_tests.csv")

## 7. Gate Contribution Analysis

Quantify the contribution of each gate to overall performance.

In [ ]:
# Calculate performance drop when each gate is removed
full_sensitivity = ablation_summary[ablation_summary['Configuration'] == 'Full SRGL']['Sensitivity'].values[0]
full_specificity = ablation_summary[ablation_summary['Configuration'] == 'Full SRGL']['Specificity'].values[0]

gate_contributions = []

single_gate_removals = {
    'G1 (Critical Flags)': 'Without G1 (Critical)',
    'G2 (Moderate Risk)': 'Without G2 (Moderate)',
    'G3 (Data Quality)': 'Without G3 (Data Quality)',
    'G4 (TiTrATE)': 'Without G4 (TiTrATE)',
    'G5 (Uncertainty)': 'Without G5 (Uncertainty)',
    'G6 (Temporal)': 'Without G6 (Temporal)'
}

for gate_name, config_name in single_gate_removals.items():
    config_data = ablation_summary[ablation_summary['Configuration'] == config_name]
    
    if len(config_data) > 0:
        sensitivity_drop = full_sensitivity - config_data['Sensitivity'].values[0]
        specificity_drop = full_specificity - config_data['Specificity'].values[0]
        
        gate_contributions.append({
            'Gate': gate_name,
            'Sensitivity_Drop': sensitivity_drop,
            'Specificity_Drop': specificity_drop,
            'Average_Drop': (sensitivity_drop + specificity_drop) / 2
        })

# Create DataFrame
contribution_df = pd.DataFrame(gate_contributions).sort_values('Average_Drop', ascending=False)

print("\n" + "="*80)
print("GATE CONTRIBUTION ANALYSIS")
print("="*80)
print(contribution_df.to_string(index=False))
print("="*80)

# Visualize contributions
fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(contribution_df))
width = 0.35

bars1 = ax.bar(x - width/2, contribution_df['Sensitivity_Drop'].values * 100, 
               width, label='Sensitivity Drop', color='steelblue')
bars2 = ax.bar(x + width/2, contribution_df['Specificity_Drop'].values * 100, 
               width, label='Specificity Drop', color='seagreen')

ax.set_xlabel('Gate', fontsize=12)
ax.set_ylabel('Performance Drop (%)', fontsize=12)
ax.set_title('Gate Contribution to Overall Performance', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(contribution_df['Gate'].values, rotation=45, ha='right')
ax.legend()
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('../figures/gate_contributions.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nFigure saved: figures/gate_contributions.png")

## 8. Merging Strategy Comparison

In [ ]:
# Compare different merging strategies with full gates
print("\nComparing merging strategies...")

merging_results = {}
merging_strategies = ['conservative', 'average', 'weighted']

for strategy in merging_strategies:
    print(f"Testing {strategy} merging...")
    results = run_ablation_experiment(test_cases, 
                                     ABLATION_CONFIGS['Full SRGL'], 
                                     merging_strategy=strategy)
    merging_results[strategy] = results

# Calculate metrics for each strategy
merging_metrics = []

for strategy, results in merging_results.items():
    y_true = results['ground_truth'].values
    y_pred = results['predicted'].values
    
    metrics = SafetyMetrics(y_true, y_pred)
    
    sensitivity = metrics.sensitivity_critical(critical_threshold=3)
    specificity = metrics.specificity_safe(safe_threshold=1)
    fnr = metrics.false_negative_rate(critical_threshold=3)
    abstention = metrics.abstention_rate()
    
    merging_metrics.append({
        'Strategy': strategy.capitalize(),
        'Sensitivity': sensitivity['sensitivity'],
        'Specificity': specificity['specificity'],
        'FNR': fnr['fnr'],
        'Abstention': abstention['abstention_rate']
    })

merging_df = pd.DataFrame(merging_metrics)

print("\n" + "="*80)
print("MERGING STRATEGY COMPARISON")
print("="*80)
print(merging_df.to_string(index=False))
print("="*80)

# Visualize comparison
fig, ax = plt.subplots(figsize=(10, 6))

x = np.arange(len(merging_df))
width = 0.2

ax.bar(x - 1.5*width, merging_df['Sensitivity'].values, width, label='Sensitivity', color='steelblue')
ax.bar(x - 0.5*width, merging_df['Specificity'].values, width, label='Specificity', color='seagreen')
ax.bar(x + 0.5*width, merging_df['FNR'].values, width, label='FNR', color='coral')
ax.bar(x + 1.5*width, merging_df['Abstention'].values, width, label='Abstention', color='mediumpurple')

ax.set_xlabel('Merging Strategy', fontsize=12)
ax.set_ylabel('Metric Value', fontsize=12)
ax.set_title('Performance by Merging Strategy', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(merging_df['Strategy'].values)
ax.legend()
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('../figures/merging_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nFigure saved: figures/merging_comparison.png")

## 9. Save Comprehensive Results

In [ ]:
# Save all ablation results
ablation_summary.to_csv('../results/ablation_study_summary.csv', index=False)
contribution_df.to_csv('../results/gate_contributions.csv', index=False)
merging_df.to_csv('../results/merging_comparison.csv', index=False)

print("\n" + "="*80)
print("ABLATION STUDY COMPLETE")
print("="*80)
print("\nResults saved:")
print("  results/ablation_study_summary.csv")
print("  results/gate_contributions.csv")
print("  results/merging_comparison.csv")
print("  results/ablation_significance_tests.csv")
print("\nFigures saved:")
print("  figures/ablation_study_comprehensive.png")
print("  figures/gate_contributions.png")
print("  figures/merging_comparison.png")
print("="*80)

## Summary

### Key Findings:

1. **Full SRGL Performance**: The complete 6-gate system with conservative merging achieves the best balance of sensitivity, specificity, and safe abstention.

2. **Critical Gates**:
   - G1 (Critical Flags): Essential for high sensitivity
   - G5 (Uncertainty): Significant contribution to safe abstention
   - G3 (Data Quality): Important for maintaining reliability

3. **Novel vs Traditional**:
   - Novel gates (G3, G5, G6) provide substantial improvement over traditional gates alone
   - Traditional gates (G1, G2, G4) form a strong baseline

4. **Merging Strategies**:
   - Conservative merging: Best overall safety profile
   - Average/Weighted: Lower abstention but higher FNR

5. **Statistical Significance**: McNemar's tests with Bonferroni correction confirm that removing key gates significantly degrades performance.